# 🌐 Модуль 4 · Веб-основи — як працює веб (для розуміння REST API)

Перш ніж будувати API, треба розуміти **середовище**, у якому воно живе — веб. Цей зошит пояснює
«велику картину»: як браузер/застосунок спілкується із сервером, що таке DNS, HTTP/HTTPS, і що
робити, коли звичайного «запит-відповідь» замало (WebSocket, Webhooks). Ці теми — **обов'язкова
частина будь-якої співбесіди на бекенд**.

**Передумова:** [Урок 1 — REST та HTTP-методи](lektsiya-1-django-rest-framework-osnovy.ipynb).

### Що ви зрозумієте
- модель **клієнт-сервер** і повний **шлях запиту** (що стається, коли вводиш URL);
- що таке **DNS**, **HTTP** і чим від нього відрізняється **HTTPS**;
- будову **URL**, запиту та відповіді;
- що таке **WebSocket** і **Webhooks** і коли вони замість REST.

## 1. Клієнт-сервер: хто з ким говорить

Веб побудований на моделі **клієнт-сервер**:
- **Клієнт** — той, хто **просить**: браузер, мобільний застосунок, інший сервіс, `curl`.
- **Сервер** — той, хто **відповідає**: ваш Django-бекенд.

Спілкування — це завжди **запит → відповідь** (request → response). Клієнт надсилає запит, сервер
обробляє й повертає відповідь. Сервер сам, без запиту, клієнту нічого не шле (це важливо — див.
розділ 7 про WebSocket/Webhooks).

```
   КЛІЄНТ                                СЕРВЕР
  (браузер) ──── HTTP-запит  ─────────▶ (Django)
            ◀─── HTTP-відповідь ───────
```

## 2. Що відбувається, коли ви вводите URL

Ви написали `https://example.com/articles?page=2` і натиснули Enter. Насправді стається ціла
низка кроків:

```
1. DNS      example.com  ──▶  IP-адреса (напр. 93.184.216.34)      «де сервер?»
2. TCP      встановлення з'єднання із сервером за цією IP          «алло, з'єднаймось»
3. TLS      (для https) рукостискання + шифрування                «говорімо таємно»
4. HTTP     клієнт шле запит: GET /articles?page=2                 «дай мені це»
5. Сервер   обробляє (Django: urls -> view -> ...) і формує відповідь
6. HTTP     сервер шле відповідь: 200 OK + дані (HTML/JSON)        «ось, тримай»
7. Клієнт   показує результат (рендерить сторінку / парсить JSON)
```

### Будова URL
```
  https :// example.com : 443 / articles ? page=2 #top
  └─┬─┘     └────┬────┘  └┬┘  └───┬────┘ └──┬──┘ └┬┘
 схема       хост      порт   шлях     query  фрагмент
(протокол)  (домен)          (ресурс) (параметри)
```

In [ ]:
# Розберемо URL на частини стандартним модулем (без мережі):
from urllib.parse import urlparse, parse_qs

url = "https://example.com:443/articles?page=2&sort=new#top"
p = urlparse(url)
print("схема (протокол):", p.scheme)     # https
print("хост (домен)   :", p.hostname)    # example.com
print("порт           :", p.port)        # 443
print("шлях (ресурс)  :", p.path)        # /articles
print("query-параметри:", parse_qs(p.query))  # {'page': ['2'], 'sort': ['new']}
print("фрагмент       :", p.fragment)    # top

## 3. DNS — «телефонна книга» інтернету

Комп'ютери знаходять одне одного за **IP-адресами** (напр. `93.184.216.34`), а люди пам'ятають
**доменні імена** (`example.com`). **DNS (Domain Name System)** — служба, що перетворює домен на IP.

```
браузер: "який IP у example.com?"
   │
   ▼
DNS-резолвер ──▶ корневі сервери ──▶ сервери .com (TLD) ──▶ авторитетний сервер example.com
   │                                                              │
   ◀──────────────────  93.184.216.34  ◀──────────────────────────
```

Відповіді **кешуються** (у резолвера, ОС, браузера), тому повторний перехід не питає DNS щоразу.

> 🎯 **Питання: що таке DNS?** Це система, що перекладає доменні імена на IP-адреси (як контакт у
> телефоні → номер). Без неї довелося б пам'ятати IP кожного сайту.

## 4. HTTP — мова вебу

**HTTP (HyperText Transfer Protocol)** — текстовий протокол «запит-відповідь». Він **stateless**
(без стану): кожен запит самодостатній, сервер не пам'ятає попередніх (тому автентифікацію
надсилають у кожному запиті — згадайте токени з Уроку 2).

**Запит** складається з: рядка (метод + шлях), **заголовків** (headers) і **тіла** (body):
```
POST /api/posts/ HTTP/1.1
Host: example.com
Authorization: Token 9a8b7c...          ← заголовки (метадані запиту)
Content-Type: application/json

{"title": "Привіт", "body": "..."}       ← тіло (дані)
```

**Відповідь** — рядок статусу, заголовки й тіло:
```
HTTP/1.1 201 Created
Content-Type: application/json

{"id": 1, "title": "Привіт"}
```

Часті заголовки: `Content-Type` (формат тіла), `Authorization` (хто ти), `Accept` (який формат
хочу у відповідь), `User-Agent` (клієнт), `Cache-Control` (кешування).

## 5. 🔎 HTTP проти HTTPS

**HTTPS = HTTP + TLS** (шифрування). Різниця критична:

| | HTTP | HTTPS |
|-|------|-------|
| Порт | 80 | 443 |
| Шифрування | ❌ (текст видно всім по дорозі) | ✅ (TLS/SSL) |
| Захист від підслуховування/підміни | ❌ | ✅ |
| Сертифікат | не треба | треба (від центру сертифікації, CA) |
| Значок у браузері | «Не захищено» | 🔒 замок |

Через **HTTP** дані (паролі, токени!) йдуть **відкритим текстом** — будь-хто «посередині» (Wi-Fi,
провайдер) може їх прочитати. **HTTPS** шифрує канал через **TLS-рукостискання**: сервер надає
**сертифікат**, сторони узгоджують ключі, далі весь трафік зашифрований.

> 🎯 **Питання: у чому різниця HTTP і HTTPS?** HTTPS — це той самий HTTP, але поверх **TLS**:
> трафік **зашифровано**, сервер підтверджує свою справжність **сертифікатом**. У проді API завжди
> має бути на HTTPS — інакше токени й паролі можна перехопити.

## 6. HTTP-методи та коди статусу (нагадування)

Це серце REST — детально в [Уроці 1](lektsiya-1-django-rest-framework-osnovy.ipynb):
- **методи:** `GET` (читати), `POST` (створити), `PUT`/`PATCH` (оновити), `DELETE` (видалити),
  а також `HEAD`/`OPTIONS` (службові);
- **коди:** `2xx` успіх, `3xx` перенаправлення, `4xx` винен клієнт, `5xx` винен сервер.

Швидка функція-класифікатор для наочності:

In [ ]:
def classify_status(code):
    groups = {1: "інформаційний", 2: "успіх", 3: "перенаправлення",
              4: "помилка клієнта", 5: "помилка сервера"}
    return groups.get(code // 100, "невідомо")

for code in [200, 201, 301, 404, 401, 500]:
    print(f"{code} -> {classify_status(code)}")

## 7. 🔎 Поза межами «запит-відповідь»: WebSocket і Webhooks

REST/HTTP працює за схемою **клієнт запитав — сервер відповів** (pull). А що, коли потрібні
**миттєві оновлення** (чат, сповіщення, ціни), і клієнт не знає, коли питати? Є два підходи.

### WebSocket — постійний двобічний канал
Замість багатьох окремих запитів клієнт **один раз** відкриває **постійне** з'єднання
(`ws://` / `wss://`), і далі **обидві** сторони можуть слати дані **будь-коли**. Ідеально для
**реального часу**: чати, онлайн-ігри, живі стрічки, нотифікації.

```
REST (polling):  клієнт ──"є новини?"──▶ сервер   (і так щосекунди — марно)
                 клієнт ◀──"нема"──────

WebSocket:       клієнт ══════ постійний канал ══════ сервер
                 сервер сам штовхає дані, щойно вони з'явились ▶
```

### Webhooks — «сервер сам вам подзвонить»
**Webhook** — це коли **зовнішній сервіс сам робить HTTP-запит до ВАШОГО URL**, щойно стається
подія. Це «зворотний API» (server-to-server callback). Приклад: платіжка (Stripe) після оплати
шле `POST` на ваш `/webhooks/payment/` — вам не треба її опитувати.

```
Звичайний API:  ВИ ──запит──▶ сервіс      (ви ініціюєте)
Webhook:        сервіс ──подія (POST)──▶ ВАШ URL   (сервіс ініціює)
```

| | Опитування (polling) | WebSocket | Webhook |
|-|----------------------|-----------|---------|
| Хто ініціює | клієнт, постійно | обидва | зовнішній сервер |
| Зв'язок | багато HTTP-запитів | одне постійне | один HTTP на подію |
| Для чого | простий «час від часу» | реальний час (чат) | події між сервісами (оплата) |

> 🎯 **Питання: WebSocket vs Webhook?** WebSocket — **постійний двобічний** канал для real-time
> між клієнтом і сервером (чат). Webhook — **разовий HTTP-запит**, який **зовнішній сервіс** шле
> на ваш URL при події (оплата пройшла). WebSocket — про «живе з'єднання», webhook — про
> «повідом мене, коли станеться».

## 8. 🚀 Middle+: REST та альтернативи

REST — не єдиний спосіб будувати API:

| Стиль | Ідея | Коли |
|-------|------|------|
| **REST** | ресурси + HTTP-методи, JSON | стандарт, простота, найпоширеніший |
| **GraphQL** | клієнт сам описує, які поля хоче (один ендпоінт) | коли клієнтам треба гнучко/різні набори полів |
| **gRPC** | бінарний протокол (protobuf) поверх HTTP/2, дуже швидкий | зв'язок між мікросервісами |

Junior будує **REST**; про GraphQL/gRPC досить знати, що вони існують і навіщо.

## 9. 🎯 Що спитають на співбесіді (веб-основи)

| Питання | Коротка відповідь |
|---------|-------------------|
| Що таке API? | Інтерфейс, через який програми обмінюються даними; REST API — поверх HTTP |
| Клієнт-сервер? | Клієнт просить (запит), сервер відповідає; сервер сам не ініціює |
| Що таке DNS? | Перетворює доменне ім'я на IP-адресу |
| HTTP vs HTTPS? | HTTPS = HTTP + TLS (шифрування + сертифікат сервера) |
| HTTP stateless — що це? | Кожен запит самодостатній; сервер не пам'ятає попередніх → auth у кожному запиті |
| HTTP-методи? | GET/POST/PUT/PATCH/DELETE (+HEAD/OPTIONS); safe/idempotent |
| Коди статусу? | 2xx успіх, 3xx редирект, 4xx клієнт, 5xx сервер |
| WebSocket? | Постійний двобічний канал для real-time (чат, нотифікації) |
| Webhook? | Зовнішній сервіс сам шле HTTP-запит на ваш URL при події (оплата) |
| Що стається при вводі URL? | DNS → TCP → TLS → HTTP-запит → сервер → відповідь → рендер |

# ✅ Підсумок
- **Клієнт-сервер:** клієнт шле запит — сервер відповідає (сам сервер не ініціює).
- **Шлях запиту:** URL → **DNS** (домен→IP) → TCP → **TLS** (для https) → **HTTP**-запит → сервер → відповідь.
- **HTTP** — текстовий, **stateless** протокол «запит-відповідь» (метод + заголовки + тіло).
- **HTTPS = HTTP + TLS** — шифрування + сертифікат; у проді обов'язково.
- **WebSocket** — постійний двобічний канал (real-time); **Webhook** — зовнішній сервіс шле запит вам на подію.

### ▶️ Куди далі
- [Урок 1](lektsiya-1-django-rest-framework-osnovy.ipynb) — REST, HTTP-методи, перші ендпоінти;
- [Урок 2](lektsiya-2-viewsets-routery-autentyfikatsiya.ipynb) — автентифікація й права;
- [`proyekt-django/`](proyekt-django/) — усе це на живому коді.

### 📚 Хочу знати більше
- Як працює веб (MDN): <https://developer.mozilla.org/en-US/docs/Learn/Common_questions/Web_mechanics/How_does_the_Internet_work>
- HTTP (MDN): <https://developer.mozilla.org/en-US/docs/Web/HTTP>
- Що стається, коли вводиш URL: <https://github.com/alex/what-happens-when>
- WebSocket (MDN): <https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API>